<pre>00 - Naive RAG and Enhanced Q&A generation with modified chunking procedure
Steps in this notebook:

Setup and model access check
Document preparation
Chunking
Indexing
Naive RAG
Naive Q&A generation
First evaluation
See README.md for the full design.</pre>

In [1]:
import logging
import os
import random
from pathlib import Path

from aieng.syn_data.text import (
    LLMSettings,
    OpenAICompatibleClient,
    load_implementation_dotenv,
    use_repo_root,
)
from rich.console import Console
from rich.table import Table


repo_root = use_repo_root()
load_implementation_dotenv()

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True)
console = Console(width=100)
print(f"Running from repository root: {repo_root}")

Running from repository root: /home/coder/synthetic-data-bootcamp


In [2]:
SEED = 42
random.seed(SEED)

SOURCE_DOCS_DIR = repo_root / "implementations/qa_text_generation/data/documents"
DESIGN_DIR = repo_root / "implementations/RAG_QA_design"
DATA_DIR = DESIGN_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CHUNKS_PATH = DATA_DIR / "chunks.jsonl"
NAIVE_QA_PATH = DATA_DIR / "naive_qa.jsonl"
NAIVE_RESULTS_PATH = DATA_DIR / "naive_results.jsonl"

console.print(f"Source documents: {SOURCE_DOCS_DIR}")
console.print(f"Output data dir:  {DATA_DIR}")

Source documents: 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/documents

Output data dir:  /home/coder/synthetic-data-bootcamp/implementations/RAG_QA_design/data

<pre>Model Access Check

The Vector proxy key only permits some models, so we send a tiny request to each candidate and see which ones answer. 

Edit CANDIDATE_MODELS to test other names.</pre>

In [3]:
CANDIDATE_MODELS = [
    os.getenv("TEACHER_MODEL", "gpt-4o"),
    os.getenv("JUDGE_MODEL", "gemini-3.5-flash"),
    "gpt-4o-mini",
]
CANDIDATE_MODELS = list(dict.fromkeys(CANDIDATE_MODELS))


def check_model_access(model: str) -> tuple[bool, str]:
    """Send a tiny request and report whether the proxy allows the model (reasoning models need token headroom)."""
    client = OpenAICompatibleClient(LLMSettings.from_env(model_var="__UNSET__", default_model=model))
    try:
        reply = client.complete("Reply with the single word: ok", temperature=0.0, max_tokens=256)
    except Exception as exc:  # noqa: BLE001
        return False, str(exc)[:80]
    return True, reply.strip()[:40]


table = Table(show_header=True, header_style="bold magenta")
table.add_column("model")
table.add_column("allowed")
table.add_column("reply / error")

access = {}
for model in CANDIDATE_MODELS:
    ok, detail = check_model_access(model)
    access[model] = ok
    table.add_row(model, "yes" if ok else "NO", detail)


<pre>Step 2 - Document preparation with Enhanced Data Preparation, and revised chunking methods

CIBC credit card agreement: in scope, this is what the RAG agent answers from.

The CIBC file is hard-wrapped, so sentences are split across lines. The cleaner joins the wrapped lines, keeps list items on their own lines, and removes page numbers and repeated header/footer lines if there are any.</pre>

In [5]:
import re
from collections import Counter

from aieng.syn_data.text import list_domain_documents
from aieng.syn_data.text.documents import load_document_text


specs = {spec.doc_id: spec for spec in list_domain_documents("finance")}
raw_texts = {doc_id: load_document_text(Path(spec.local_path)) for doc_id, spec in specs.items()}

table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "role", "chars", "words", "lines"):
    table.add_column(column)
for doc_id, text in raw_texts.items():
    table.add_row(
        doc_id,
        str(specs[doc_id].role),
        str(len(text)),
        str(len(text.split())),
        str(text.count("\n") + 1),
    )
console.print(table)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃ doc_id                     ┃ role           ┃ chars ┃ words ┃ lines ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ cfpb_credit_card_agreement │ policy_dense   │ 46941 │ 7983  │ 280   │
│ sec_investor_bulletin      │ scope_boundary │ 13795 │ 2133  │ 73    │
└────────────────────────────┴────────────────┴───────┴───────┴───────┘

In [6]:
PAGE_NUMBER_RE = re.compile(r"^\s*(?:page\s+)?\d+(?:\s+(?:of|/)\s+\d+)?\s*$", re.IGNORECASE)
LIST_ITEM_RE = re.compile(r"^(?:[a-z]\.\s|-\s|\d+\.\s)")  # "a. ", "- ", "1. "
SENTENCE_END = (".", "?", "!", ":", ";", '"', "\u201d", ")")
MAX_HEADER_LEN = 60  # a short line repeated 3+ times is treated as a header or footer


def clean_text(text: str) -> tuple[str, dict[str, int]]:
    """Join hard-wrapped lines, keep list items separate, and drop page numbers and repeated headers/footers."""
    lines = [line.strip() for line in text.split("\n")]
    counts = Counter(line for line in lines if line)
    repeated = {line for line, n in counts.items() if n >= 3 and len(line) <= MAX_HEADER_LEN}

    lengths = sorted(len(line) for line in lines if line)
    wrap_width = lengths[int(len(lengths) * 0.9)]  # the typical full-line width of the source

    stats = {"page_numbers_removed": 0, "headers_footers_removed": 0, "lines_joined": 0}
    blocks: list[str] = []
    for block in re.split(r"\n\s*\n", text):
        logical: list[str] = []
        current = ""
        last_len = 0
        for line in (raw.strip() for raw in block.split("\n")):
            if not line:
                continue
            if PAGE_NUMBER_RE.match(line):
                stats["page_numbers_removed"] += 1
                continue
            if line in repeated:
                stats["headers_footers_removed"] += 1
                continue
            if not current:
                current, last_len = line, len(line)
                continue
            starts_new_line = (
                LIST_ITEM_RE.match(line) is not None
                or (current.endswith(SENTENCE_END) and last_len < 0.85 * wrap_width)
                or (last_len < 0.6 * wrap_width and line[0].isupper())  # short line, then a capital: heading
            )
            if starts_new_line:
                logical.append(current)
                current = line
            else:
                current = f"{current} {line}"
                stats["lines_joined"] += 1
            last_len = len(line)
        if current:
            logical.append(current)
        if logical:
            blocks.append("\n".join(logical))
    return "\n\n".join(blocks), stats

In [7]:
clean_texts: dict[str, str] = {}
table = Table(show_header=True, header_style="bold magenta")
for column in ("doc_id", "chars before", "chars after", "lines joined", "page numbers", "headers/footers"):
    table.add_column(column)

for doc_id, text in raw_texts.items():
    cleaned, stats = clean_text(text)
    clean_texts[doc_id] = cleaned
    table.add_row(
        doc_id,
        str(len(text)),
        str(len(cleaned)),
        str(stats["lines_joined"]),
        str(stats["page_numbers_removed"]),
        str(stats["headers_footers_removed"]),
    )
console.print(table)

sample_doc = "cfpb_credit_card_agreement"
marker = "1. Introduction"
start = raw_texts[sample_doc].index(marker)
console.print("[bold]Before (section 12):[/bold]")
console.print(raw_texts[sample_doc][start : start + 450])
start = clean_texts[sample_doc].index(marker)
console.print("\n[bold]After (section 12):[/bold]")
console.print(clean_texts[sample_doc][start : start + 450])

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ doc_id              ┃ chars before ┃ chars after ┃ lines joined ┃ page numbers ┃ headers/footers ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ cfpb_credit_card_a… │ 46941        │ 46927       │ 61           │ 0            │ 0               │
│ sec_investor_bulle… │ 13795        │ 13795       │ 2            │ 0            │ 0               │
└─────────────────────┴──────────────┴─────────────┴──────────────┴──────────────┴─────────────────┘

Before (section 12):

1. Introduction

This Agreement applies to your Account and replaces any earlier agreements for your Account. If you 
do any of the following, you are acknowledging you have received, understood and agreed to this 
Agreement:
• sign your Card
• use your Card
• have a Balance
• activate a Card
• access or make a payment on your Account
b) When any Cardholder uses your Account to make a Purchase or makes any other charges to your 
Account, we are loan

After (section 12):

1. Introduction

This Agreement applies to your Account and replaces any earlier agreements for your Account. If you 
do any of the following, you are acknowledging you have received, understood and agreed to this 
Agreement:
• sign your Card • use your Card • have a Balance • activate a Card • access or make a payment on 
your Account b) When any Cardholder uses your Account to make a Purchase or makes any other charges 
to your Account, we are loan

In [72]:
from aieng.syn_data.text import (
    DEFAULT_TEST_PARAS_PER_DOC,
    Paragraph,
    ParagraphSplit,
    load_typed_jsonl,
    save_typed_jsonl,
)
from aieng.syn_data.text.documents import paragraphs_from_document, sample_test_paragraphs

In [76]:
specs = {spec.doc_id: spec for spec in list_domain_documents("finance")}
print(specs)

{'cfpb_credit_card_agreement': DocumentSpec(doc_id='cfpb_credit_card_agreement', title='CFPB Sample Credit Card Agreement', role=<DocumentRole.POLICY_DENSE: 'policy_dense'>, domain='finance', source_url='https://files.consumerfinance.gov/f/documents/201401_cfpb_credit-card-agreement_english.pdf', local_path='/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/documents/cfpb_credit_card_agreement.txt'), 'sec_investor_bulletin': DocumentSpec(doc_id='sec_investor_bulletin', title='SEC Investor Bulletin', role=<DocumentRole.SCOPE_BOUNDARY: 'scope_boundary'>, domain='finance', source_url='https://www.sec.gov/files/ib_fraud.pdf', local_path='/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/documents/sec_investor_bulletin.txt')}


<pre>Step 3 - Enhanced Chunking Process

This step reuses the existing qa_text_generation chunking, applied to the cleaned text: CIBC documents needs a lot of cleaning and preprocessing

paragraphs_from_document splits each document on blank lines and merges pieces shorter than min_chars into their neighbour.

sample_test_paragraphs holds out DEFAULT_TEST_PARAS_PER_DOC chunks per document, by chunk, with a fixed seed.

Chunk IDs have the form doc_id::pNNNN.

Nothing is trained in this project, so the two splits are development (diagnose failures and tune) and test (frozen, run once at the end). The package's ParagraphSplit enum only has train and test, so in memory ParagraphSplit.TRAIN stands for development. The saved file uses the names development and test.

Chunks are saved to data/chunks.jsonl, one JSON object per line.</pre>

In [77]:
import re
from pathlib import Path

def extract_pages(path):
    text = Path(path).read_text(encoding="utf-8")

    pages = []
    for index, page_text in enumerate(text.split("\f")):
        page_text = re.sub(r"[ \t]+", " ", page_text)
        page_text = re.sub(r"\n{3,}", "\n\n", page_text).strip()

        pages.append({
            "page": index + 1,
            "text": page_text,
        })

    return pages


pages = extract_pages('implementations/qa_text_generation/data/documents/cfpb_credit_card_agreement.txt')

print(
    f"Pages: {len(pages)} | "
    f"Nonempty: {sum(bool(p['text']) for p in pages)}"
)

Pages: 1 | Nonempty: 1


In [321]:

import re
import pandas as pd


def clean_text(text: str) -> str:
    # Remove bullet symbols.
    text = re.sub(r"[•●▪◦‣⁃]\s*", "", text)

    # Remove section headings such as "1. Introduction" and "9.1. Provisions".
    text = re.sub(
        r"(?<!\w)\d+(?:\.\d+)*\.\s+",
        "",
        text,
    )

    # Remove ordered-list markers such as "1. billed interest"
    # and "2. Installment Plan payments".
    text = re.sub(
        r"(?<!\w)\d+\.\s+(?=[A-Za-z“])",
        "",
        text,
    )

    # Remove lettered clause/list markers such as "a)", "b)", and "c."
    text = re.sub(
        r"(?<!\w)[a-zA-Z][.)]\s+(?=[A-Z“a-z])",
        "",
        text,
    )
    
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"[ \t]+", " ", text).strip()

    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text

def chunk_page(text, size=2000, overlap=200):
    assert 0 <= overlap < size

    text = clean_agreement_text(text)

    if not text:
        return []

    # Preserve paragraph boundaries.
    paragraphs = re.split(r"\n\s*\n", text)
    units = []

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if not paragraph:
            continue

        if len(paragraph) <= size:
            units.append(paragraph)
        else:
            # Split long paragraphs at sentence boundaries.
            sentences = re.split(r"(?<=[.!?])\s+", paragraph)
            current = ""

            for sentence in sentences:
                if len(sentence) > size:
                    if current:
                        units.append(current)
                        current = ""

                    # Hard-split unusually long sentences.
                    for i in range(0, len(sentence), size):
                        units.append(sentence[i:i + size])
                    continue

                candidate = f"{current} {sentence}".strip()

                if len(candidate) <= size:
                    current = candidate
                else:
                    if current:
                        units.append(current)
                    current = sentence

            if current:
                units.append(current)

    # Pack units into chunks, adding overlap where possible.
    result = []
    current = ""

    for unit in units:
        candidate = f"{current}\n\n{unit}".strip()

        if len(candidate) <= size:
            current = candidate
        else:
            if current:
                result.append(current)

            prefix = ""

            if result and overlap:
                previous = result[-1]
                overlap_start = max(0, len(previous) - overlap)

                boundary = max(
                    previous.rfind(". ", overlap_start),
                    previous.rfind("? ", overlap_start),
                    previous.rfind("! ", overlap_start),
                    previous.rfind("\n", overlap_start),
                )

                if boundary >= overlap_start:
                    prefix = previous[boundary + 1:].strip()
                else:
                    prefix = previous[-overlap:].strip()

            current = f"{prefix}\n\n{unit}".strip()

            # Ensure the chunk doesn't exceed the character limit.
            if len(current) > size:
                current = current[-size:]

    if current:
        result.append(current)

    return [c for c in result if c.strip()]

for page in pages:
    page["text"] = clean_text(page["text"])

# Generate chunks from the original page text.
chunks = []
chunk_id = 0

for page in pages:
    for text in chunk_page(
        page["text"],
        size=2400,
        overlap=350,
    ):
        chunks.append({
            "page": page["page"],
            "chunk_id": chunk_id,
            "context": text,
        })
        chunk_id += 1

print("Chunks:", len(chunks))

# Inspect chunk sizes and previews.
chunk_df = pd.DataFrame([
    {
        "page": chunk["page"],
        "chunk_id": chunk["chunk_id"],
        "characters": len(chunk["context"]),
        "words": len(chunk["context"].split()),
        "preview": chunk["context"][:150].replace("\n", " "),
    }
    for chunk in chunks
])

for chunk in chunks:
    chunk["context"] = re.sub(
        r"\s+", " ", chunk["context"]
    ).strip()

print(chunk_df.to_string(index=False))


Chunks: 21
 page  chunk_id  characters  words                                                                                                                                                preview
    1         0        2304    395 CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and replaces any earlier agreements for your Account. If you do any of t
    1         1        2267    409 You must not keep any written or electronic record of your PIN.  If you choose a PIN, you must not use all or any part of a number or word that can be
    1         2        2400    403 ansaction or your Account in our discretion for any reason.  You must not use your Account: for any fraudulent or illegal purpose, including buying il
    1         3        2400    420 t, the CIBC Debit Agreement applies to any losses.  If you use CIBC Online Banking but do not comply with the CIBC Electronic Access Agreement, you ma
    1         4        2400    434 t or Cash Limit.  We may

In [328]:
# Convert dictionaries to your existing Paragraph schema.
# Replace doc_id with the actual document ID from your specs if needed.
# Convert your existing raw_chunks into Paragraph objects.
doc_id = next(iter(specs))

doc_id = next(iter(specs))

final_chunks = [
    Paragraph(
        doc_id=doc_id,
        index=chunk["chunk_id"],
        para_id=f"{doc_id}::p{chunk['chunk_id']:04d}",
        text=chunk["context"],
        role=DocumentRole.POLICY_DENSE,
        split=DEVELOPMENT,
    )
    for chunk in raw_chunks
]

# Assign test and development splits.
test_chunks, development_chunks = sample_test_paragraphs(
    final_chunks,
    n_test=5,
    seed=SEED,
)

chunks = test_chunks + development_chunks
chunks.sort(key=lambda chunk: (chunk.doc_id, chunk.index))

# Save with the original serialization.
save_typed_jsonl(CHUNKS_PATH, chunks, to_dict=chunk_to_record)

console.print(
    f"Saved {len(chunks)} chunks: "
    f"{sum(p.split is ParagraphSplit.TEST for p in chunks)} test, "
    f"{sum(p.split is DEVELOPMENT for p in chunks)} development"
)

Saved 18 chunks: 5 test, 13 development

In [330]:
# Reload from disk to confirm the saved file round-trips.
chunks = load_typed_jsonl(CHUNKS_PATH, chunk_from_record)

# Remove actual newline characters from each chunk's text
for chunk in chunks:
    chunk.text = re.sub(r"\s+", " ", chunk.text).strip()

# Save the cleaned chunks back to disk
save_typed_jsonl(CHUNKS_PATH, chunks, to_dict=chunk_to_record)

PosixPath('/home/coder/synthetic-data-bootcamp/implementations/RAG_QA_design/data/chunks.jsonl')

In [331]:

# Reload from disk to confirm the saved file round-trips.
chunks = load_typed_jsonl(CHUNKS_PATH, chunk_from_record)

train_paragraphs = [
    p for p in chunks
    if p.split is DEVELOPMENT
]

table = Table(show_header=True, header_style="bold magenta")

for column in (
    "doc_id", "chunks", "test", "development",
    "min chars", "mean chars", "max chars"
):
    table.add_column(column)

for doc_id in specs:
    doc = [chunk for chunk in chunks if chunk.doc_id == doc_id]

    if not doc:
        console.print(f"[yellow]No chunks found for {doc_id}[/yellow]")
        continue

    sizes = [len(chunk.text) for chunk in doc]

    table.add_row(
        doc_id,
        str(len(doc)),
        str(sum(chunk.split is ParagraphSplit.TEST for chunk in doc)),
        str(sum(chunk.split is DEVELOPMENT for chunk in doc)),
        str(min(sizes)),
        str(round(sum(sizes) / len(sizes))),
        str(max(sizes)),
    )

console.print(table)

for chunk in chunks[:2]:
    console.print(
        f"\n[bold]{chunk.para_id}[/bold] "
        f"role={chunk.role.value} "
        f"split={'development' if chunk.split is DEVELOPMENT else 'test'} "
        f"index={chunk.index} chars={len(chunk.text)}"
    )
    console.print(chunk.text)

console.print("\n[bold]Raw line in chunks.jsonl:[/bold]")
lines = CHUNKS_PATH.read_text(encoding="utf-8").splitlines()

if lines:
    console.print(lines[0][:400] + " ...")


No chunks found for sec_investor_bulletin

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ doc_id                     ┃ chunks ┃ test ┃ development ┃ min chars ┃ mean chars ┃ max chars ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ cfpb_credit_card_agreement │ 18     │ 5    │ 13          │ 1203      │ 2636       │ 2799      │
└────────────────────────────┴────────┴──────┴─────────────┴───────────┴────────────┴───────────┘

cfpb_credit_card_agreement::p0000 role=policy_dense split=development index=0 chars=2659

CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and replaces any 
earlier agreements for your Account. If you do any of the following, you are acknowledging you have 
received, understood and agreed to this Agreement: sign your Card use your Card have a Balance 
activate a Card access or make a payment on your Account When any Cardholder uses your Account to 
make a Purchase or makes any other charges to your Account, we are loaning that amount to the 
Primary Cardholder. The Primary Cardholder must repay the Balance as described in this Agreement, 
without setoff or counterclaim. All Cardholders must comply with this Agreement, but only the 
Primary Cardholder is required to repay the Balance. The Primary Cardholder is responsible for all 
of the obligations under this Agreement, including the obligations of any Authorized Users and any 
Installment Plans requested on the Account by any Authorized Users. Other documents and agreements 
Your Summary of Rates and Fees is part of this Agreement. The CIBC Debit Agreement applies when you 
use your Card and PIN to access any non-credit card account(attached to your Card. It does not apply
when you use your Account. The CIBC Electronic Access Agreement applies when you use CIBC Online 
Banking or CIBC Mobile Banking to access your Account. If there is a conflict between this Agreement
and the CIBC Debit Agreement or CIBC Electronic Access Agreement, this Agreement will govern to the 
extent necessary to resolve the conflict. If you use a Mobile Payment App, our terms of use for that
app also apply. You can get a current copy of any of these documents at www.cibc.com or by calling 
CIBC Credit Card Services at the number on the back of your Card. Your rights and responsibilities 
Use Your Account. You may use your Account as long as you comply with this Agreement and we allow 
you to. You are authorizing a Transaction if you show you agree to it by any means acceptable to 
CIBC. Cards and Convenience Cheques belong to CIBC. You must return them if we ask you to. Keep Your
PIN Safe. You must keep your PIN(confidential. You must never disclose your PIN to anyone. If we 
send you a PIN, you must memorize it and then destroy the document right away. You must not keep any
written or electronic record of your PIN. If you choose a PIN, you must not use all or any part of a
number or word that can be easily found or guessed, such as: your or a relative’s name, birth date 
or telephone number a number on any of your other cards, accounts or ID cards that you keep with or 
near your Card Keep Your Account Safe. You must take reasonable steps to keep your Account safe.

cfpb_credit_card_agreement::p0001 role=policy_dense split=development index=1 chars=2354

You must take reasonable steps to keep your Account safe. At a minimum, this means you will: sign 
and activate your Card as soon as you get it never let a person who is not a Cardholder use your 
Card or your Account if you give your Card Details to a website or app, make sure other people can’t
access that website or app to make purchases with your Card Details tell us right away if you see 
any unauthorized Transactions, if your Card or Card Details are lost or stolen or if you think that 
someone knows your PIN Manage Your Pre-Authorized Payments. You can set up pre- authorized payments 
with a merchant. You must give the merchant the information they need, including any updates to that
information. We are not responsible if any payments cannot be posted. If you want to stop any 
pre-authorized payment, you must contact the merchant first and then check that they followed your 
instructions. After you have contacted the merchant to cancel your pre-authorized payment, you can 
contact us for further assistance if you need it. Review Your Statements. The Primary Cardholder 
must make sure to review their statement each month. You must look for Transactions that you do not 
recognize or did not authorize. If you find an error or problem, you must tell us within thirty days
of the Statement Date. After that time, you may not make a claim for any Transaction on that 
statement. Pay the Minimum Payment. We must receive at least the Minimum Payment by the payment due 
date each month, even if the Primary Cardholder does not receive a statement or the statement is 
late. Follow Mobile Payment App Terms. If you have registered your Card or Account in a Mobile 
Payment App, you must comply with our terms of use. Do Not Use Your Account Improperly. We may block
Transactions that we can identify as internet gambling. We may block any Transaction or your Account
in our discretion for any reason. You must not use your Account: for any fraudulent or illegal 
purpose, including buying illegal goods or services to pay any amount that you owe us Liability for 
loss The maximum liability of a Primary Cardholder for unauthorized Transactions, including if a 
Card is lost or stolen, is $50, unless a Cardholder has demonstrated gross negligence or, in Quebec,
gross fault in safeguarding their Card, account information such as Card Details, or PIN.

Raw line in chunks.jsonl:

{"doc_id": "cfpb_credit_card_agreement", "para_id": "cfpb_credit_card_agreement::p0000", "text": 
"CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and replaces any 
earlier agreements for your Account. If you do any of the following, you are acknowledging you have 
received, understood and agreed to this Agreement: sign your Card use your Card have a Balance 
activate a C ...

<pre>
Step 4 - Embedding and indexing (semantic retrieval)

Chunks are embedded with sentence-transformers (all-MiniLM-L6-v2, 384 dimensions, normalized) and searched by cosine similarity.

Indexed: all CFPB chunks, development and test together. The split only decides which chunks questions are generated from, so test questions can still find their source chunk.
Not indexed: the SEC chunks. They are the source of out-of-scope questions, which should have nothing relevant to retrieve.

Long chunks: the model reads at most 256 tokens, and 5 of the 35 CFPB chunks are longer. Each chunk is split into overlapping windows of at most 200 tokens, each window is embedded, and a chunk is scored by its best window. Results are whole chunks with the same IDs, so the ground truth for retrieval metrics does not change.

Saved: the window vectors and the window-to-chunk map go to data/index/, so later runs load them instead of re-embedding.
</pre>

In [332]:
import hashlib
import json

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from aieng.syn_data.text.schemas import DocumentRole


EMBED_MODEL = "all-MiniLM-L6-v2"
WINDOW_TOKENS = 200  # the model reads at most 256 tokens
WINDOW_STRIDE = 160  # 40 tokens of overlap between neighbouring windows
INDEX_DIR = DATA_DIR / "index"

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device)
console.print(f"Embedding model {EMBED_MODEL} on {device}, max_seq_length={embedder.max_seq_length}")

def split_into_windows(text: str) -> list[str]:
    tokenizer = embedder.tokenizer

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )

    offsets = encoded["offset_mapping"]

    if len(offsets) <= WINDOW_TOKENS:
        return [text]

    windows = []

    for start in range(0, len(offsets), WINDOW_STRIDE):
        end = min(start + WINDOW_TOKENS, len(offsets))

        windows.append(
            text[offsets[start][0]:offsets[end - 1][1]]
        )

        if end == len(offsets):
            break

    return windows


def index_key(chunks: list[Paragraph]) -> str:
    """Fingerprint of everything the vectors depend on, so a stale saved index is never reused."""
    payload = [EMBED_MODEL, WINDOW_TOKENS, WINDOW_STRIDE] + [(chunk.para_id, chunk.text) for chunk in chunks]
    return hashlib.sha256(json.dumps(payload).encode()).hexdigest()


class SemanticIndex:
    """Window embeddings for a list of chunks. A chunk is scored by its best-matching window."""

    def __init__(self, chunks: list[Paragraph], vectors: np.ndarray, window_chunk: list[int]) -> None:
        self.chunks = list(chunks)
        self.vectors = vectors  # (n_windows, dim), L2-normalized
        self.window_chunk = np.array(window_chunk)  # window row -> chunk position

    @classmethod
    def build(cls, chunks: list[Paragraph]) -> "SemanticIndex":
        window_texts: list[str] = []
        window_chunk: list[int] = []
        for position, chunk in enumerate(chunks):
            for window in split_into_windows(chunk.text):
                window_texts.append(window)
                window_chunk.append(position)
        vectors = embedder.encode(window_texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        return cls(chunks, vectors, window_chunk)

    def save(self, directory: Path) -> None:
        directory.mkdir(parents=True, exist_ok=True)
        np.save(directory / "vectors.npy", self.vectors)
        meta = {"key": index_key(self.chunks), "model": EMBED_MODEL, "window_chunk": self.window_chunk.tolist(), "chunk_ids": [chunk.para_id for chunk in self.chunks]}
        (directory / "meta.json").write_text(json.dumps(meta), encoding="utf-8")

    @classmethod
    def load(cls, directory: Path, chunks: list[Paragraph]) -> "SemanticIndex | None":
        """Load a saved index if it matches these chunks and settings, otherwise return None."""
        meta_path = directory / "meta.json"
        if not meta_path.exists():
            return None
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta["key"] != index_key(chunks):
            return None
        return cls(chunks, np.load(directory / "vectors.npy"), meta["window_chunk"])

    def search(self, query: str, top_k: int = 3) -> list[tuple[float, Paragraph]]:
        """Return the top_k (score, chunk) pairs, best first. Score is cosine similarity of the best window."""
        query_vector = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
        window_scores = self.vectors @ query_vector
        chunk_scores = np.full(len(self.chunks), -1.0)
        np.maximum.at(chunk_scores, self.window_chunk, window_scores)
        order = np.argsort(-chunk_scores, kind="stable")[:top_k]
        return [(float(chunk_scores[position]), self.chunks[position]) for position in order]


indexed_chunks = [chunk for chunk in chunks if chunk.role is DocumentRole.POLICY_DENSE]
index = SemanticIndex.load(INDEX_DIR, indexed_chunks)
if index is None:
    index = SemanticIndex.build(indexed_chunks)
    index.save(INDEX_DIR)
    source = "built and saved"
else:
    source = "loaded from disk"
console.print(f"Index {source}: {len(indexed_chunks)} chunks, {len(index.vectors)} windows, dim {index.vectors.shape[1]}")

2026-10-29 22:20:20,568 INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-10-29 22:20:20,573 INFO sentence_transformers.base.model: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3523.65it/s]
2026-10-29 22:20:20,701 INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-29 22:20:20,742 INFO httpx2: HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-10-29 22:20:20,844 INFO httpx2: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/2_Normalize/config.json "HTTP/1.1 404 Not Found"


Embedding model all-MiniLM-L6-v2 on cpu, max_seq_length=256

Index loaded from disk: 18 chunks, 69 windows, dim 384

In [333]:
SANITY_QUERIES = [
    "What is the late payment fee?",
    "When will I not be charged a finance charge on new purchases?",
    "How do I dispute a billing error on my statement?",
    "What happens if I am in default?",
    "How should I protect my online investment account from fraud?",
]

# A saved index must give the same results as a freshly built one.
fresh = SemanticIndex.build(indexed_chunks)
reloaded = SemanticIndex.load(INDEX_DIR, indexed_chunks)
assert reloaded is not None
for query in SANITY_QUERIES:
    assert [c.para_id for _, c in fresh.search(query)] == [c.para_id for _, c in reloaded.search(query)], query
console.print("Saved index matches a fresh build on all sanity queries.")

table = Table(show_header=True, header_style="bold magenta")
for column in ("query", "rank", "chunk", "split", "score", "preview"):
    table.add_column(column)
for query in SANITY_QUERIES:
    for rank, (score, chunk) in enumerate(index.search(query, top_k=3), start=1):
        table.add_row(
            query if rank == 1 else "",
            str(rank),
            chunk.para_id.split("::")[1],
            "development" if chunk.split is DEVELOPMENT else "test",
            f"{score:.3f}",
            chunk.text[:60].replace("\n", " "),
        )
console.print(table)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (543 > 256). Running this sequence through the model will result in indexing errors


Saved index matches a fresh build on all sanity queries.

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ query                        ┃ rank ┃ chunk ┃ split       ┃ score ┃ preview                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ What is the late payment     │ 1    │ p0004 │ test        │ 0.479 │ The statement may cover the  │
│ fee?                         │      │       │             │       │ whole three months or only   │
│                              │      │       │             │       │ the l                        │
│                              │ 2    │ p0001 │ development │ 0.460 │ You must take reasonable     │
│                              │      │       │             │       │ steps to keep your Account   │
│                              │      │       │             │       │ safe. At                     │
│                              │ 3    │ p0017 │ development │ 0.457 │ Minimum Payment means the    │
│                              │      │       │             │       │ minimum amount that must be  │
│                              │      │       │             │       │ paid o                       │
│ When will I not be charged a │ 1    │ p0006 │ development │ 0.588 │ rder of the payment          │
│ finance charge on new        │      │       │             │       │ allocation for an Account.   │
│ purchases?                   │      │       │             │       │ We have not s                │
│                              │ 2    │ p0004 │ test        │ 0.509 │ The statement may cover the  │
│                              │      │       │             │       │ whole three months or only   │
│                              │      │       │             │       │ the l                        │
│                              │ 3    │ p0003 │ development │ 0.503 │ The branch selling rate is a │
│                              │      │       │             │       │ foreign exchange rate we     │
│                              │      │       │             │       │ set. W                       │
│ How do I dispute a billing   │ 1    │ p0008 │ development │ 0.503 │ with your new Card. There    │
│ error on my statement?       │      │       │             │       │ are other optional Services  │
│                              │      │       │             │       │ that y                       │
│                              │ 2    │ p0003 │ development │ 0.503 │ The branch selling rate is a │
│                              │      │       │             │       │ foreign exchange rate we     │
│                              │      │       │             │       │ set. W                       │
│                              │ 3    │ p0001 │ development │ 0.497 │ You must take reasonable     │
│                              │      │       │             │       │ steps to keep your Account   │
│                              │      │       │             │       │ safe. At                     │
│ What happens if I am in      │ 1    │ p0013 │ test        │ 0.299 │ Governing Law. This          │
│ default?                     │      │       │             │       │ Agreement will be governed   │
│                              │      │       │             │       │ by and interp                │
│                              │ 2    │ p0012 │ test        │ 0.270 │ use or keep your Account     │
│                              │      │       │             │       │ after the effective date of  │
│                              │      │       │             │       │ the cha                      │
│                              │ 3    │ p0014 │ development │ 0.270 │ at 1 800 465-Any complaint   │
│                              │      │       │             │       │ not resolved within 14 days  │
│                              │      │       │             │       │ of be                        │
│ How should I protect my      │ 1    │ p0001 │ development │

<pre>

Step 5 - Naive RAG

One answering function: embed the question, retrieve the top 5 chunks from the semantic index, and answer from them only.

Model roles (different families so the judge does not favour its own answers): </pre>

In [335]:
from dataclasses import dataclass, field


def make_client(model_var: str, default_model: str) -> OpenAICompatibleClient:
    return OpenAICompatibleClient(LLMSettings.from_env(model_var=model_var, default_model=default_model))


generator_client = make_client("RAG_GENERATOR_MODEL", "gpt-4o")
answer_client = make_client("RAG_ANSWER_MODEL", "gemini-3.5-flash")
judge_client = make_client("RAG_JUDGE_MODEL", "gpt-4o")
for role, client in ("generator", generator_client), ("answerer", answer_client), ("judge", judge_client):
    console.print(f"{role:10s} {client.settings.model}")

TOP_K = 5
ANSWER_MAX_TOKENS = 4096  # reasoning models spend 300-2000 tokens thinking before the answer; 1024 truncated answers
REFUSAL_TEXT = "Not specified in the agreement."
# Both replies the answer prompt uses for questions the agreement cannot answer.
REFUSAL_STARTS = ("not specified in the agreement", "please refer to the protection section")

ANSWER_SYSTEM_PROMPT = """
you are a credit card agreement question answering assistant. you must answer the question only based on the provided numbered context below.

# Output Rules:
* Always answer based on the provided context.
* For protection-related questions, especially regarding online investment protection, banking account privacy, or fraud, reply with: "Please refer to the protection section of your online investment for guidance."
* Do not use outside knowledge or guess.
* End with a line of the form 'Sources: <chunk ids you used>'.
* If the answer is not in the provided context, reply exactly: "Not specified in the agreement. Please contact your card issuer for more information at 123@gmail.com."
* If the question is protection related, especially regarding online investment protection or banking account privacy or fraud, you must reply with:
"Please refer to the protection section of your online investment for guidance."
* You can only provide guidance on credit card agreements.
* Never mention the "the document says" or "the passage suggests" in your answer.
* Be polite, and maintain a professional tone in your response.
"""

@dataclass
class RagResult:
    question: str
    answer: str
    retrieved_ids: list[str] = field(default_factory=list)
    retrieved_scores: list[float] = field(default_factory=list)
    model: str = ""

    @property
    def refused(self) -> bool:
        return self.answer.strip().lower().startswith(REFUSAL_STARTS)


def format_context(context_chunks: list[Paragraph]) -> str:
    return "\n\n".join(f"[{chunk.para_id.split('::')[1]}]\n{chunk.text}" for chunk in context_chunks)


def is_complete(answer: str) -> bool:
    """A finished answer ends with a Sources line, or is a refusal. Anything else was cut off."""
    text = answer.strip().lower()
    return text.startswith(REFUSAL_STARTS) or text.splitlines()[-1].startswith("sources") if text else False


def rag_answer(question: str, *, top_k: int = TOP_K) -> RagResult:
    """Retrieve the top-k chunks for the question and answer from them."""
    retrieved = index.search(question, top_k=top_k)
    prompt = f"Context passages:\n\n{format_context([chunk for _, chunk in retrieved])}\n\nQuestion: {question}"
    for _ in range(3):  # the proxy sometimes returns a cut-off answer, so ask again
        answer = answer_client.complete(prompt, system=ANSWER_SYSTEM_PROMPT, temperature=0.0, max_tokens=ANSWER_MAX_TOKENS)
        if is_complete(answer):
            break
    return RagResult(
        question=question,
        answer=answer.strip(),
        retrieved_ids=[chunk.para_id for _, chunk in retrieved],
        retrieved_scores=[round(score, 4) for score, _ in retrieved],
        model=answer_client.settings.model,
    )

generator  gpt-4o

answerer   gemini-3.5-flash

judge      gpt-4o

In [336]:
for query in SANITY_QUERIES:
    result = rag_answer(query)
    ids = ", ".join(f"{rid.split('::')[1]} ({score:.2f})" for rid, score in zip(result.retrieved_ids, result.retrieved_scores, strict=True))
    console.print(f"[bold]Q:[/bold] {query}")
    console.print(f"[bold]Retrieved:[/bold] {ids}")
    console.print(f"[bold]A:[/bold] {result.answer}\n")

Q: What is the late payment fee?

Retrieved: p0004 (0.48), p0001 (0.46), p0017 (0.46), p0009 (0.43), p0008 (0.40)

A: Not specified in the agreement. Please contact your card issuer for more information at 
123@gmail.com.

Q: When will I not be charged a finance charge on new purchases?

Retrieved: p0006 (0.59), p0004 (0.51), p0003 (0.50), p0009 (0.49), p0007 (0.47)

...

Retrieved: p0013 (0.30), p0012 (0.27), p0014 (0.27), p0011 (0.26), p0000 (0.26)

A: If actions are taken to collect your Balance when you are in default:

* **Actions to collect your Balance:** We may debit any deposit account you have with us, or redeem 
all or part of any fixed-term deposit you have with us (whether matured or not) and apply those 
funds toward your Balance.
* **Legal fees:** To the fullest extent permitted by law, you will pay all legal fees and expenses 
incurred to collect your Balance (this step is not applicable in Quebec).
* **For Quebec residents:** Before enforcing this clause, you must be sent a written notice and, 
unless exempted, a statement of account. Within 30 days of receiving the notice and statement of 
account (where necessary), you may either remedy the default or present an application to the court 
to change the contract's payment terms and conditions.

Sources: p0011

Q: How should I protect my online investment account from fraud?

Retrieved: p0001 (0.46), p0014 (0.43), p0000 (0.42), p0002 (0.36), p0010 (0.31)

A: Please refer to the protection section of your online investment for guidance.

Sources:

<pre>

Step 6 - Naive Q&A generation
The generator (gpt-4o) gets one chunk and writes questions with gold answers.

In scope: 3 questions for each of the 25 development CFPB chunks (the 10 test chunks are not used for generation). The prompt asks for a mix of simple lookups and questions that need several facts, in customer wording, with the supporting evidence copied from the chunk.

Out of scope: 1 question for each of the 11 development SEC chunks, written as a credit card customer asking something the agreement cannot answer. The gold answer is fixed to the refusal text. This gives roughly 13% out-of-scope questions (the README target is 15-25%, so we may raise this to 2 per chunk).

Checks: evidence must appear in the chunk, then the package's heuristic filters run. No judge or support check yet.
Saved: data/naive_qa.jsonl, with the source chunk ID, split, and prompt version on every item.
The prompts are in the next cell so they are easy to edit. Change PROMPT_VERSION when you change them.</pre>

In [337]:
PROMPT_VERSION = "naive-v1"
IN_SCOPE_N = 3  # questions per CFPB chunk
OUT_OF_SCOPE_N = 1  # questions per SEC chunk
GEN_TEMPERATURE = 0.3

GENERATOR_SYSTEM = "You write exam questions for evaluating a document question answering system."

IN_SCOPE_PROMPT = '''You are writing exam questions about a consumer credit card agreement.
Read the passage and write {n} questions that a customer could ask.

Rules:
- Every question must be fully answerable from this passage alone.
- Write the way a real customer would. Do not copy phrases from the passage, and do not mention "the passage" or "the document".
- Mix difficulty: at least one simple lookup, and at least one question that needs two or more facts, conditions or exceptions from the passage.
- The gold answer must be complete and use only facts stated in the passage.
- "evidence" must be the exact sentence or sentences copied from the passage that support the answer.

Return JSON: {"questions": [{"question": "...", "gold_answer": "...", "evidence": "..."}]}

Passage:
{passage}'''

OUT_OF_SCOPE_PROMPT = '''You are writing exam questions to test whether a credit card assistant knows its limits.
The assistant only answers questions about a consumer credit card agreement. The passage below comes from a different document (investor education material).
Write {n} question that a credit card customer might plausibly ask, about the topic of the passage, that a credit card agreement could not answer.

Rules:
- The question must be about the topic of the passage (for example investing, brokerage accounts, online account fraud), phrased as a customer asking their credit card provider.
- Write the way a real customer would. Do not mention "the passage" or "the document".
- Do not include an answer.

Return JSON: {"questions": [{"question": "..."}]}

Passage:
{passage}'''


def build_generation_prompt(chunk: Paragraph) -> str:
    in_scope = chunk.role is DocumentRole.POLICY_DENSE
    template = IN_SCOPE_PROMPT if in_scope else OUT_OF_SCOPE_PROMPT
    n = IN_SCOPE_N if in_scope else OUT_OF_SCOPE_N
    return template.replace("{n}", str(n)).replace("{passage}", chunk.text)


console.print(build_generation_prompt(indexed_chunks[3])[:1400])

You are writing exam questions about a consumer credit card agreement.
Read the passage and write 3 questions that a customer could ask.

Rules:
- Every question must be fully answerable from this passage alone.
- Write the way a real customer would. Do not copy phrases from the passage, and do not mention "the
passage" or "the document".
- Mix difficulty: at least one simple lookup, and at least one question that needs two or more 
facts, conditions or exceptions from the passage.
- The gold answer must be complete and use only facts stated in the passage.
- "evidence" must be the exact sentence or sentences copied from the passage that support the 
answer.

Return JSON: {"questions": [{"question": "...", "gold_answer": "...", "evidence": "..."}]}

Passage:
The branch selling rate is a foreign exchange rate we set. We may earn money on the difference 
between the buying and selling prices of currency, which is in addition to any fee you pay us on a 
foreign currency Transaction or any fees you pay us for your Account. Currency conversion may not 
happen on the day of the Transaction. If you get a foreign currency credit to your Account, it will 
be converted at the exchange rate in effect when the credit is applied. This exchange rate may be 
different than the exchange that applied to the original purchase. If you use your Card to get a 
cash withdrawal from a non-credit card account

In [338]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_SYNTHETIC_TARGET_SIZE,
    PARAGRAPHS_PATH,
    SYNTHETIC_IBT_RAW_PATH,
    SYNTHETIC_RAW_PATH,
    Paragraph,
    ParagraphSplit,
    QASample,
    compare_generation_strategies,
    create_teacher_client,
    effective_synthetic_target,
    generate_grounded_training_corpus,
    generate_raw_synthetic_corpus,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    use_repo_root,
)
from rich.console import Console
from rich.panel import Panel
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

Notebook path is already the root path: /home/coder/synthetic-data-bootcamp


<pre> Step 7 Stage 1: Add the Synthetic Generation from the chunks</pre>

In [339]:
# Reload from disk to confirm the saved file round-trips.
chunks = load_typed_jsonl(CHUNKS_PATH, chunk_from_record)

In [340]:
console.print(f"Train paragraphs available: {len(chunks)}")

para = chunks[0]

table = Table(title="Train Paragraph [1]", show_lines=True)
table.add_column("Field", style="bold cyan")
table.add_column("Value", style="white")

table.add_row("doc_id", str(para.doc_id))
table.add_row("para_id", str(para.para_id))
table.add_row("role", str(para.role))
table.add_row("split", str(para.split))
table.add_row("index", str(para.index))
table.add_row("text", para.text if len(para.text) < 500 else para.text[:500] + " ...")

console = Console()
console.print(Panel(table, title="Train Paragraph Overview"))

Train paragraphs available: 18

╭─────────────────────────────────────────── Train Paragraph Overview ────────────────────────────────────────────╮
│                                               Train Paragraph [1]                                               │
│ ┏━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Field   ┃ Value                                                                                             ┃ │
│ ┡━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ doc_id  │ cfpb_credit_card_agreement                                                                        │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ para_id │ cfpb_credit_card_agreement::p0000                                                                 │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ role    │ policy_dense                                                                                      │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ split   │ train                                                                                             │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ index   │ 0                                                                                                 │ │
│ ├─────────┼───────────────────────────────────────────────────────────────────────────────────────────────────┤ │
│ │ text    │ CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and replaces any    │ │
│ │         │ earlier agreements for your Account. If you do any of the following, you are acknowledging you    │ │
│ │         │ have received, understood and agreed to this Agreement: sign your Card use your Card have a       │ │
│ │         │ Balance activate a Card access or make a payment on your Account When any Cardholder uses your    │ │
│ │         │ Account to make a Purchase or makes any other charges to your Account, we are loaning that amount │ │
│ │         │ to the Primary Cardholder ...                                                                     │ │
│ └─────────┴───────────────────────────────────────────────────────────────────────────────────────────────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

<pre> Step 7, Stage 2: Rank the existing chunks based on word count and quality scores </pre>

In [341]:
import re
import pandas as pd

def score_chunk(text):
    text = text.strip()
    words = text.split()
    issues = []

    if not text:
        return 0, 0, "empty"

    if len(words) < 30:
        issues.append("short")

    if text[0].islower():
        issues.append("lowercase_start")

    if text[-1] not in ".!?\":)]}":
        issues.append("possible_incomplete_ending")

    if re.search(r"\s{3,}", text):
        issues.append("excess_whitespace")

    score = 100
    if len(words) < 30:
        score -= 20
    if text[0].islower():
        score -= 10
    if text[-1] not in ".!?\":)]}":
        score -= 10
    if re.search(r"\s{3,}", text):
        score -= 10

    return len(words), max(0, score), ", ".join(issues)


records = []

for i, paragraph in enumerate(chunks):
    text = paragraph.text.strip()

    word_count, quality_score, issues = score_chunk(text)

    records.append({
        "chunk_id": f"p{i:04d}",
        "text": text,
        "word_count": word_count,
        "quality_score": quality_score,
        "issues": issues,
    })

chunk_scores = pd.DataFrame(records)

# Show chunks needing the most review.
chunk_scores = chunk_scores.sort_values(
    "quality_score",
    ascending=True,
)

print(
    chunk_scores[
        ["chunk_id", "word_count", "quality_score", "issues"]
    ].head(30).to_string(index=False)
)



chunk_id  word_count  quality_score          issues
   p0002         470             90 lowercase_start
   p0007         464             90 lowercase_start
   p0006         446             90 lowercase_start
   p0008         467             90 lowercase_start
   p0012         460             90 lowercase_start
   p0014         451             90 lowercase_start
   p0011         476             90 lowercase_start
   p0016         438             90 lowercase_start
   p0004         482            100                
   p0005         447            100                
   p0001         409            100                
   p0003         505            100                
   p0010         453            100                
   p0009         484            100                
   p0013         420            100                
   p0000         466            100                
   p0015         461            100                
   p0017         190            100                


<pre> Step 7 Stage 3: Generate the first level Synthetic - using the generate_raw_synthetic_Corpus, which will generate 2 questions per paragragh </pre>

In [342]:
raw_samples = generate_raw_synthetic_corpus(
    generator_client,
    chunks,
    max_paragraphs=len(chunks),
    questions_per_para=2,
)
console.print(f"Raw synthetic samples: {len(raw_samples)}")

save_typed_jsonl(
    SYNTHETIC_RAW_PATH,
    raw_samples,
    to_dict=lambda sample: sample.to_dict(),
)
console.print("Raw synthetic samples saved to", SYNTHETIC_RAW_PATH)

Raw synthetic samples: 90

Raw synthetic samples saved to 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/synthetic_raw.jsonl

In [343]:
target_size = effective_synthetic_target(
    chunks,
    requested=DEFAULT_SYNTHETIC_TARGET_SIZE,
    one_per_paragraph=True,
)
console.print(f"[bold green]Back-translation generation target:[/bold green] {target_size} (cap = # train paragraphs)")

grounded_candidates = generate_grounded_training_corpus(
    generator_client,
    chunks,
    target_size=target_size,
    min_overlap=0.15,
)
console.print(f"Generated {len(grounded_candidates)} back-translated candidates")

save_typed_jsonl(
    SYNTHETIC_IBT_RAW_PATH,
    grounded_candidates,
    to_dict=lambda sample: sample.to_dict(),
)
console.print("Raw IBT samples saved to", SYNTHETIC_IBT_RAW_PATH)
grounded_candidates[0].metadata if grounded_candidates else {}

Back-translation generation target: 18 (cap = # train paragraphs)

Generated 18 back-translated candidates

Raw IBT samples saved to 
/home/coder/synthetic-data-bootcamp/implementations/qa_text_generation/data/synthetic/synthetic_back_translation_ra
w.jsonl

{'generation_strategy': 'instruction_backtranslation',
 'grounding_overlap': 1.0}

In [344]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_JUDGE_THRESHOLD,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_IBT_PATH,
    SYNTHETIC_IBT_RAW_PATH,
    SYNTHETIC_RAW_PATH,
    QASample,
    apply_heuristic_filters,
    create_judge_client,
    filter_with_judge,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_heuristic_rejections,
    summarize_judge_scores,
    use_repo_root,
    write_json,
)
from rich import box
from rich.console import Console
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

Notebook path is already the root path: /home/coder/synthetic-data-bootcamp


<pre> Step 7 Stage 4: Apply the first level of Heursitic filters and LLM as a judge filter </pre>

In [345]:
raw_samples = load_typed_jsonl(SYNTHETIC_RAW_PATH, QASample.from_dict)
kept, rejected = apply_heuristic_filters(raw_samples)
print(f"Kept {len(kept)} / {len(raw_samples)} samples")

reason_counts = summarize_heuristic_rejections(rejected)
print("Rejection reasons:", reason_counts)

# Inspect a few rejected samples and which heuristic(s) fired
table = Table(title="Sample heuristic rejections", show_lines=True)
table.add_column("id", style="cyan", max_width=20)
table.add_column("reason(s)", style="red")
table.add_column("question", overflow="fold")
for row in rejected[:8]:
    table.add_row(row.get("id", ""), row.get("reasons", row.get("reason", "")), row.get("question", ""))
console.print(table)

Kept 81 / 90 samples
Rejection reasons: {'duplicate_question': 9}


                                    Sample heuristic rejections                                     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id                   ┃ reason(s)          ┃ question                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 583776ba-ac75-4cfd-… │ duplicate_question │ How does the CIBC Cardholder Agreement prioritize    │
│                      │                    │ its terms over the CIBC Debit Agreement and CIBC     │
│                      │                    │ Electronic Access Agr                                │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ c5f6645f-b787-44b9-… │ duplicate_question │ Under what circumstances is the Primary Cardholder   │
│                      │                    │ responsible for a transaction and any resulting      │
│                      │                    │ interest, fees, and l                                │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ d32bc405-c207-4750-… │ duplicate_question │ How does the payment allocation process differ for   │
│                      │                    │ residents outside Quebec compared to Quebec          │
│                      │                    │ residents when a payment                             │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 34a5e960-08f4-4f27-… │ duplicate_question │ How does the payment allocation process differ for   │
│                      │                    │ residents outside Quebec compared to Quebec          │
│                      │                    │ residents when a payment                             │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ dee414ae-c643-4635-… │ duplicate_question │ How does the payment allocation process prioritize   │
│                      │                    │ different types of balances and interest rates when  │
│                      │                    │ a payment is made                                    │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 443ee30a-e679-40ab-… │ duplicate_question │ How is interest calculated on balances with          │
│                      │                    │ different annual interest rates, such as Purchases,  │
│                      │                    │ Cash Advances, Balance T                             │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 685fb1f6-65b1-4607-… │ duplicate_question │ What are the conditions under which a Cardholder's   │
│                      │                    │ Card can be automatically enrolled in Click to Pay,  │
│                      │                    │ and what happens                                     │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 3c013088-f47e-43f7-… │ duplicate_question │ What are the implications for a Primary Cardholder   │
│                      │                    │ if their Account is not in Good Standing, and how    │
│                      │                    │ does it affect thei                                  │
└──────────────────────┴────────────────────┴──────────────────────────────────────────────────────┘

In [346]:
from IPython.display import HTML
from IPython.display import display as ipy_display


judge = make_client("RAG_JUDGE_MODEL", "gemini-3.5-flash")

heuristic_rejected = rejected
state = {"done": 0}
total = len(kept)
handle = ipy_display(
    HTML(f"<b>Judge scoring</b> 0/{total} <progress value='0' max='{total}' style='width:24em'></progress>"),
    display_id=True,
)


def on_progress() -> None:
    """Refresh the in-notebook judge scoring progress bar."""
    state["done"] += 1
    done = state["done"]
    handle.update(
        HTML(
            f"<b>Judge scoring</b> {done}/{total} <progress value='{done}' max='{total}' style='width:24em'></progress>",
        ),
    )


filtered_samples, judge_scores, judge_rejected = filter_with_judge(
    judge,
    kept,
    threshold=DEFAULT_JUDGE_THRESHOLD,
    on_progress=on_progress,
)

# filter_with_judge re-runs heuristics on `kept`, so keep the original heuristic
# rejections and append judge-only rejects for an accurate quality report.
rejected = heuristic_rejected + [row for row in judge_rejected if row.get("reason") == "below_judge_threshold"]
console.print(
    f"[bold green]After judge filter:[/bold green] [yellow]{len(filtered_samples)}[/yellow] kept, "
    f"[red]{len(rejected)}[/red] rejected "
    f"[dim]({len(heuristic_rejected)} heuristic + "
    f"{len(rejected) - len(heuristic_rejected)} judge)[/dim]",
)
summarize_judge_scores(judge_scores)

2026-10-29 22:25:04,042 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 9b240b37-0919-4714-935d-b00b82ae2010
2026-10-29 22:25:05,483 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 87f7e2d3-560d-405d-8df2-8c79591ffb3d
2026-10-29 22:25:06,700 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 167befff-b11d-401e-9481-3ceb68b740e4
2026-10-29 22:25:07,888 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 96db5eed-7a2b-47c1-adf4-c9a4ca1ee686
2026-10-29 22:25:09,566 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: ca9c866e-6b2e-4771-bdeb-7d346e5e6a69
2026-10-29 22:25:10,772 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: b2f7602f-d7f0-4d8a-b60d-ee1ab5a5ef8c
2026-10-29 22:25:12,431 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: e0af7772-9d5d-4c84-9c2f-0d41e875e1d0
2026-10-29 22:25:13,826 INFO aieng.syn_data.text.judge:

After judge filter: 81 kept, 9 rejected (9 heuristic + 0 judge)

{'correctness': 4.888888888888889,
 'coherence': 5.0,
 'instruction_following': 5.0,
 'factual_plausibility': 4.888888888888889,
 'average': 4.944444444444445}

In [347]:
from collections import Counter

console.print(f"Total filtered questions: {len(filtered_samples)}")

split_counts = Counter(str(sample.split) for sample in filtered_samples)
console.print(f"Questions by split: {dict(split_counts)}")

samples_to_generate = [
    sample for sample in filtered_samples
    if sample.split is DEVELOPMENT
]

console.print(f"Questions selected for generation: {len(samples_to_generate)}")

Total filtered questions: 81

Questions by split: {'train': 59, 'test': 22}

Questions selected for generation: 59

<pre> Step 7 - Stage 5: Game Changer - The second layer of Synthetic Generation (this time from the existing set of questions) </pre>

In [348]:
from concurrent.futures import ThreadPoolExecutor
from aieng.syn_data.text import QASample


def split_label(sample: QASample) -> str:
    return "development" if sample.split is DEVELOPMENT else "test"


def build_generation_prompt(sample: QASample) -> str:
    return f"""
Context:
{sample.context}

Existing question:
{sample.question}

Existing gold answer:
{sample.gold_answer}

Generate NEW questions based on the context.

Requirements:
- Each question must be distinct and answerable from the context.
- For every in-scope question, provide a non-empty, correct gold_answer.
- The evidence field MUST be an exact, verbatim substring copied from the context.
- Do not paraphrase the evidence.
- Do not repeat the existing question.
- Return fewer questions only if the context genuinely cannot support more.

Return JSON with a "questions" array containing objects with:
- "question"
- "gold_answer"
- "evidence"
"""


def generate_for_sample(sample: QASample) -> list[QASample]:
    in_scope = sample.metadata.get(
        "in_scope",
        sample.role is DocumentRole.POLICY_DENSE,
    )

    payload = generator_client.complete_json(
        build_generation_prompt(sample),
        system=GENERATOR_SYSTEM,
        temperature=GEN_TEMPERATURE,
        max_tokens=2048,
    )

    generated = []

    for k, item in enumerate(payload.get("questions", [])):
        question = str(item.get("question", "")).strip()
        if not question:
            continue

        gold_answer = (
            str(item.get("gold_answer", "")).strip()
            if in_scope
            else REFUSAL_TEXT
        )

        # Reject in-scope questions without a valid answer.
        if in_scope and not gold_answer:
            continue

        evidence = (
            str(item.get("evidence", "")).strip()
            if in_scope
            else ""
        )

        metadata = {
            **sample.metadata,
            "in_scope": in_scope,
            "source_sample_id": sample.id,
            "source_chunk_id": (
                sample.metadata.get("source_chunk_id") or sample.para_id
                if in_scope
                else None
            ),
            "evidence": evidence,
            "split_name": split_label(sample),
            "prompt_version": PROMPT_VERSION,
            "generator_model": generator_client.settings.model,
        }

        generated.append(
            QASample(
                id=f"naive-{sample.id}-{k}",
                question=question,
                gold_answer=gold_answer,
                context=sample.context,
                doc_id=sample.doc_id,
                para_id=sample.para_id,
                role=sample.role,
                instruction=sample.instruction,
                split=sample.split,
                metadata=metadata,
            )
        )

    return generated


def safe_generate(
    sample: QASample,
) -> tuple[list[QASample], str | None]:
    error = None

    for _ in range(2):
        try:
            return generate_for_sample(sample), None
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"[:150]

    return [], error


# Process filtered questions, not chunks.
samples_to_generate = [
    sample
    for sample in filtered_samples
    if sample.split is DEVELOPMENT
]

with ThreadPoolExecutor(max_workers=8) as pool:
    outcomes = list(pool.map(safe_generate, samples_to_generate))

raw_samples = [
    generated
    for generated_list, _ in outcomes
    for generated in generated_list
]

failures = [
    (sample.id, error)
    for sample, (_, error) in zip(
        samples_to_generate, outcomes, strict=True
    )
    if error
]

console.print(
    f"Generated {len(raw_samples)} questions from "
    f"{len(samples_to_generate)} input questions; "
    f"{len(failures)} question(s) failed"
)

for sample_id, error in failures:
    console.print(f"FAILED {sample_id}: {error}")

Generated 310 questions from 59 input questions; 0 question(s) failed

<pre> Step 7 - Stage 6: Quering and filtering needs more due dilligence when taken from first stage of synthetic generation </pre>

In [349]:
from collections import Counter

from aieng.syn_data.text.quality import (
    apply_heuristic_filters,
    summarize_heuristic_rejections,
)


def normalize_for_match(text: str) -> str:
    text = (
        text.replace("\u201c", '"')
        .replace("\u201d", '"')
        .replace("\u2018", "'")
        .replace("\u2019", "'")
    )
    return " ".join(text.lower().split())


# 1. Validate evidence for in-scope questions.
grounded, bad_evidence, missing_metadata = [], [], []

for sample in raw_samples:  # Validate generated samples
    in_scope = sample.metadata.get("in_scope")

    if in_scope is None or "evidence" not in sample.metadata:
        missing_metadata.append(sample)
        continue

    evidence = sample.metadata.get("evidence", "")

    if in_scope and (
        not evidence.strip()
        or normalize_for_match(evidence)
        not in normalize_for_match(sample.context)
    ):
        bad_evidence.append(sample)
    else:
        grounded.append(sample)

# 2. Apply heuristic filters.
naive_samples, rejected = apply_heuristic_filters(grounded)

console.print(
    f"raw {len(filtered_samples)} -> "
    f"missing metadata {len(missing_metadata)} -> "
    f"evidence check removed {len(bad_evidence)} -> "
    f"heuristics removed {len(rejected)} -> "
    f"kept {len(naive_samples)}"
)

console.print(
    f"heuristic rejections: {summarize_heuristic_rejections(rejected)}"
)

# 3. Save the typed JSONL.
def sample_to_record(sample: QASample) -> dict:
    record = sample.to_dict()
    record["split"] = sample.metadata["split_name"]
    return record


save_typed_jsonl(NAIVE_QA_PATH, naive_samples, to_dict=sample_to_record)

console.print(f"Saved {len(naive_samples)} questions to {NAIVE_QA_PATH}")

# 4. Summarize split and scope.
table = Table(show_header=True, header_style="bold magenta")

for column in ("split", "in scope", "out of scope", "total"):
    table.add_column(column)

counts = Counter(
    (s.metadata["split_name"], s.metadata["in_scope"])
    for s in naive_samples
)

for split_name in ("development", "test"):
    inside = counts[(split_name, True)]
    outside = counts[(split_name, False)]
    table.add_row(
        split_name,
        str(inside),
        str(outside),
        str(inside + outside),
    )

console.print(table)

# 5. Inspect examples.
examples = naive_samples[:3] + [
    s for s in naive_samples if not s.metadata["in_scope"]
][:2]

for sample in examples:
    console.print(
        f"\n[bold]{sample.id}[/bold] "
        f"in_scope={sample.metadata['in_scope']} "
        f"split={sample.metadata['split_name']}"
    )
    console.print(f"Q: {sample.question}")
    console.print(f"A: {sample.gold_answer}")

raw 81 -> missing metadata 0 -> evidence check removed 20 -> heuristics removed 44 -> kept 246

heuristic rejections: {'duplicate_question': 44}

Saved 246 questions to 
/home/coder/synthetic-data-bootcamp/implementations/RAG_QA_design/data/naive_qa.jsonl

┏━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┓
┃ split       ┃ in scope ┃ out of scope ┃ total ┃
┡━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━┩
│ development │ 246      │ 0            │ 246   │
│ test        │ 0        │ 0            │ 0     │
└─────────────┴──────────┴──────────────┴───────┘

naive-9b240b37-0919-4714-935d-b00b82ae2010-0 in_scope=True split=development

...

Q: What should a cardholder do with their PIN after receiving it from CIBC?

A: Memorize it and then destroy the document right away.

naive-9b240b37-0919-4714-935d-b00b82ae2010-2 in_scope=True split=development

Q: What happens if there is a conflict between the CIBC Cardholder Agreement and the CIBC Debit 
Agreement?

A: This Agreement will govern to the extent necessary to resolve the conflict.

In [350]:
console.print(f"raw_samples count: {len(raw_samples)}")

for sample in raw_samples[:5]:
    evidence = sample.metadata.get("evidence", "")
    context = sample.context

    console.print(f"\nID: {sample.id}")
    console.print(f"Evidence: {evidence!r}")
    console.print(f"Context starts: {context[:300]!r}")
    console.print(
        f"Evidence match: "
        f"{normalize_for_match(evidence) in normalize_for_match(context)}"
    )

raw_samples count: 310

ID: naive-9b240b37-0919-4714-935d-b00b82ae2010-0

Evidence: 'If you do any of the following, you are acknowledging you have received, understood and 
agreed to this Agreement: sign your Card use your Card have a Balance activate a Card access or make
a payment on your Account'

Context starts: 'CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and 
replaces any earlier agreements for your Account. If you do any of the following, you are 
acknowledging you have received, understood and agreed to this Agreement: sign your Card use your 
Card have a Balance activate a'

Evidence match: True

...

Evidence match: True

ID: naive-9b240b37-0919-4714-935d-b00b82ae2010-4

Evidence: 'If you choose a PIN, you must not use all or any part of a number or word that can be 
easily found or guessed, such as: your or a relative’s name, birth date or telephone number a number
on any of your other cards, accounts or ID cards that you keep with or near your Card'

Context starts: 'CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and 
replaces any earlier agreements for your Account. If you do any of the following, you are 
acknowledging you have received, understood and agreed to this Agreement: sign your Card use your 
Card have a Balance activate a'

Evidence match: True

In [351]:
console.print(f"Input samples: {len(raw_samples)}")
console.print(f"Grounded: {len(grounded)}")
console.print(f"Bad evidence: {len(bad_evidence)}")
console.print(f"Missing metadata: {len(missing_metadata)}")

assert (
    len(raw_samples)
    == len(grounded) + len(bad_evidence) + len(missing_metadata)
), "Evidence-filter counts do not add up"

Input samples: 310

Grounded: 290

Bad evidence: 20

Missing metadata: 0

In [352]:
naive_samples, rejected = apply_heuristic_filters(grounded)

console.print(
    f"raw {len(raw_samples)} -> "
    f"evidence check removed {len(bad_evidence)} -> "
    f"grounded {len(grounded)} -> "
    f"heuristics removed {len(rejected)} -> "
    f"kept {len(naive_samples)}"
)
console.print(
    f"Heuristic rejections: {summarize_heuristic_rejections(rejected)}"
)

raw 310 -> evidence check removed 20 -> grounded 290 -> heuristics removed 44 -> kept 246

Heuristic rejections: {'duplicate_question': 44}

In [353]:
#check for the missing required field
sample = grounded[2]
console.print(sample.to_dict())
console.print(sample.metadata)

{
    'id': 'naive-9b240b37-0919-4714-935d-b00b82ae2010-2',
    'question': 'What happens if there is a conflict between the CIBC Cardholder Agreement and the 
CIBC Debit Agreement?',
    'gold_answer': 'This Agreement will govern to the extent necessary to resolve the conflict.',
    'doc_id': 'cfpb_credit_card_agreement',
    'para_id': 'cfpb_credit_card_agreement::p0000',
    'context': 'CIBC CARDHOLDER AGREEMENT Introduction This Agreement applies to your Account and 
replaces any earlier agreements for your Account. If you do any of the following, you are 
acknowledging you have received, understood and agreed to this Agreement: sign your Card use your 
Card have a Balance activate a Card access or make a payment on your Account When any Cardholder 
uses your Account to make a Purchase or makes any other charges to your Account, we are loaning that
amount to the Primary Cardholder. The Primary Cardholder must repay the Balance as described in this
Agreement, without setoff or counterclaim. All Cardholders must comply with this Agreement, but only
the Primary Cardholder is required to repay the Balance. The Primary Cardholder is responsible for 
all of the obligations under this Agreement, including the obligations of any Authorized Users and 
any Installment Plans requested on the Account by any Authorized Users. Other documents and 
agreements Your Summary of Rates and Fees is part of this Agreement. The CIBC Debit Agreement 
applies when you use your Card and PIN to access any non-credit card account(attached to your Card. 
It does not apply when you use your Account. The CIBC Electronic Access Agreement applies when you 
use CIBC Online Banking or CIBC Mobile Banking to access your Account. If there is a conflict 
between this Agreement and the CIBC Debit Agreement or CIBC Electronic Access Agreement, this 
Agreement will govern to the extent necessary to resolve the conflict. If you use a Mobile Payment 
App, our terms of use for that app also apply. You can get a current copy of any of these documents 
at www.cibc.com or by calling CIBC Credit Card Services at the number on the back of your Card. Your
rights and responsibilities Use Your Account. You may use your Account as long as you comply with 
this Agreement and we allow you to. You are authorizing a Transaction if you show you agree to it by
any means acceptable to CIBC. Cards and Convenience Cheques belong to CIBC. You must return them if 
we ask you to. Keep Your PIN Safe. You must keep your PIN(confidential. You must never disclose your
PIN to anyone. If we send you a PIN, you must memorize it and then destroy the document right away. 
You must not keep any written or electronic record of your PIN. If you choose a PIN, you must not 
use all or any part of a number or word that can be easily found or guessed, such as: your or a 
relative’s name, birth date or telephone number a number on any of your other cards, accounts or ID 
cards that you keep with or near your Card Keep Your Account Safe. You must take reasonable steps to
keep your Account safe.',
    'failure_mode': None,
    'role': 'policy_dense',
    'instruction': '',
    'split': 'train',
    'metadata': {
        'generation_strategy': 'zero_shot',
        'in_scope': True,
        'source_sample_id': '9b240b37-0919-4714-935d-b00b82ae2010',
        'source_chunk_id': 'cfpb_credit_card_agreement::p0000',
        'evidence': 'If there is a conflict between this Agreement and the CIBC Debit Agreement or 
CIBC Electronic Access Agreement, this Agreement will govern to the extent necessary to resolve the 
conflict.',
        'split_name': 'development',
        'prompt_version': 'naive-v1',
        'generator_model': 'gpt-4o'
    }
}

{
    'generation_strategy': 'zero_shot',
    'in_scope': True,
    'source_sample_id': '9b240b37-0919-4714-935d-b00b82ae2010',
    'source_chunk_id': 'cfpb_credit_card_agreement::p0000',
    'evidence': 'If there is a conflict between this Agreement and the CIBC Debit Agreement or CIBC 
Electronic Access Agreement, this Agreement will govern to the extent necessary to resolve the 
conflict.',
    'split_name': 'development',
    'prompt_version': 'naive-v1',
    'generator_model': 'gpt-4o'
}

<pre>
Stage 8 - Evaluate the RAG

The RAG answers every development question, then two evaluators score it.

Ranking metrics (in-scope questions only). The gold chunk is the chunk the question was written from.

Recall@3 / Recall@5: share of questions whose gold chunk is in the top 3 / top 5 retrieved.
MRR: average of 1 / rank of the gold chunk (0 if it was not retrieved).
Quality judge (gpt-4o). It sees the RAG system prompt (its instructions and prescribed replies), the question, the RAG answer and the retrieved chunks. It does not see the gold answer. For each of the three criteria it writes its reasoning first, then a true/false flag. Every answer is judged, because the protection rule applies to out-of-scope questions. Factuality and citation accuracy are reported on in-scope answers only.

Factuality: every claim in the answer is supported by the retrieved chunks.
Citation accuracy: the chunk IDs on the Sources: line are retrieved chunks that really support the answer.
Instruction following: the answer obeys all six rules below. The judge also lists the violated rule numbers.
Rule	Requirement
R1	Protection-related questions (online investment protection, banking account privacy, fraud) get exactly: "Please refer to the protection section of your online investment for guidance."
R2	An answer built from the context ends with a Sources: <chunk ids you used> line (the two exact replies are exempt)
R3	If the answer is not in the context, reply exactly: "Not specified in the agreement. Please contact your card issuer for more information at 123@gmail.com."
R4	Guidance on credit card agreements only
R5	Never say "the document says" or "the passage suggests"
R6	Polite, professional tone

Also reported: ROUGE-L against the gold answer, and the refusal rate on the out-of-scope questions (a plain text check, no judge).
</pre>

In [354]:
from concurrent.futures import ThreadPoolExecutor

EVAL_SPLIT = "development"

eval_samples = [
    s for s in naive_samples
    if s.metadata.get("split_name") == EVAL_SPLIT
]

console.print(f"Total naive questions: {len(naive_samples)}")
console.print(f"Evaluation split: {EVAL_SPLIT}")
console.print(f"Questions selected for evaluation: {len(eval_samples)}")


def safe_answer(sample: QASample) -> tuple[RagResult | None, str | None]:
    error = None

    for _ in range(2):
        try:
            return rag_answer(sample.question), None
        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"[:150]

    return None, error


with ThreadPoolExecutor(max_workers=8) as pool:
    answered = list(pool.map(safe_answer, eval_samples))


rows = []
failures = []

for sample, (result, error) in zip(eval_samples, answered, strict=True):
    if result is None:
        failures.append((sample.id, error))
        console.print(f"  FAILED {sample.id}: {error}")
        continue

    rows.append(
        {
            "id": sample.id,
            "question": sample.question,
            "in_scope": sample.metadata["in_scope"],
            "gold_chunk": sample.metadata.get("source_chunk_id"),
            "gold_answer": sample.gold_answer,
            "answer": result.answer,
            "refused": result.refused,
            "retrieved_ids": result.retrieved_ids,
            "retrieved_scores": result.retrieved_scores,
        }
    )

console.print(
    f"Answered {len(rows)} of {len(eval_samples)} questions "
    f"({sum(r['in_scope'] for r in rows)} in scope, "
    f"{sum(not r['in_scope'] for r in rows)} out of scope)"
)
console.print(f"Failed: {len(failures)}")

Total naive questions: 246

Evaluation split: development

Questions selected for evaluation: 246

Answered 246 of 246 questions (246 in scope, 0 out of scope)

Failed: 0

<pre>
Step 9 - Evaluate the RAG

The RAG answers every development question, then two evaluators score it.

Ranking metrics (in-scope questions only). The gold chunk is the chunk the question was written from.

Recall@3 / Recall@5: share of questions whose gold chunk is in the top 3 / top 5 retrieved.
MRR: average of 1 / rank of the gold chunk (0 if it was not retrieved).

Quality judge (gpt-4o). It sees the RAG system prompt (its instructions and prescribed replies), the question, the RAG answer and the retrieved chunks. It does not see the gold answer. For each of the three criteria it writes its reasoning first, then a true/false flag. Every answer is judged, because the protection rule applies to out-of-scope questions. Factuality and citation accuracy are reported on in-scope answers only.

Factuality: every claim in the answer is supported by the retrieved chunks.

Citation accuracy: the chunk IDs on the Sources: line are retrieved chunks that really support the answer.

Instruction following: the answer obeys all six rules below. The judge also lists the violated rule numbers.
</pre>

In [355]:
from statistics import mean

KS = (3, 5)


def gold_rank(row: dict) -> int | None:
    """1-based rank of the gold chunk, or None if unavailable/not retrieved."""
    gold_chunk = row.get("gold_chunk")
    ids = row.get("retrieved_ids") or []

    if not gold_chunk:
        return None

    return ids.index(gold_chunk) + 1 if gold_chunk in ids else None


def ranking_metrics(scored_rows: list[dict]) -> dict[str, float]:
    if not scored_rows:
        return {**{f"Recall@{k}": 0.0 for k in KS}, "MRR": 0.0}

    ranks = [row["gold_rank"] for row in scored_rows]

    metrics = {
        f"Recall@{k}": mean(
            1.0 if rank is not None and rank <= k else 0.0
            for rank in ranks
        )
        for k in KS
    }
    metrics["MRR"] = mean(
        1.0 / rank if rank is not None else 0.0
        for rank in ranks
    )

    return metrics


in_scope_rows = [row for row in rows if row["in_scope"]]

# Only evaluate rows with a known gold chunk.
rankable_rows = [
    row for row in in_scope_rows
    if row.get("gold_chunk")
]

for row in rankable_rows:
    row["gold_rank"] = gold_rank(row)

ranking = ranking_metrics(rankable_rows)

table = Table(show_header=True, header_style="bold magenta")
table.add_column(f"Ranking ({len(rankable_rows)} rankable in-scope questions)")
table.add_column("value")

for name, value in ranking.items():
    table.add_row(name, f"{value:.3f}")

console.print(table)
console.print(
    f"Excluded {len(in_scope_rows) - len(rankable_rows)} in-scope questions "
    "because their gold chunk ID was missing."
)

for row in rankable_rows:
    if row["gold_rank"] is None:
        gold_chunk = row["gold_chunk"]
        console.print(
            f"[bold]Gold chunk not retrieved:[/bold] "
            f"{gold_chunk.split('::')[-1]}  Q: {row['question']}"
        )

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Ranking (246 rankable in-scope questions) ┃ value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ Recall@3                                  │ 0.870 │
│ Recall@5                                  │ 0.963 │
│ MRR                                       │ 0.757 │
└───────────────────────────────────────────┴───────┘

Excluded 0 in-scope questions because their gold chunk ID was missing.

Gold chunk not retrieved: p0001  Q: What must be done if an error or problem is found on a 
statement?

Gold chunk not retrieved: p0003  Q: Who is responsible if an Authorized User goes over their 
Spending Limit?

Gold chunk not retrieved: p0016  Q: Who are considered Cardholders under this agreement?

...

Gold chunk not retrieved: p0016  Q: What does the term 'Good Standing' mean in this agreement?

Gold chunk not retrieved: p0016  Q: What are the components of Card Details?

Gold chunk not retrieved: p0016  Q: What does Click to Pay allow a Cardholder to do?

Gold chunk not retrieved: p0016  Q: What is the meaning of 'Good Standing' in this agreement?

Gold chunk not retrieved: p0017  Q: What is excluded from being considered a Purchase according to 
the passage?

LLM as a judge

In [356]:
from concurrent.futures import ThreadPoolExecutor

JUDGE_SYSTEM = (
    "You are a strict evaluator of answers produced by a document "
    "question answering system. Follow the judging criteria exactly. "
    "Return only valid JSON."
)

JUDGE_PROMPT = '''You are given the instructions a RAG system was told to follow, a question, the answer the system produced, and the document chunks it retrieved.
Each chunk starts with its ID in square brackets.
Judge the answer using ONLY the retrieved chunks and the RAG instructions. You are not given a reference answer, and you must not use outside knowledge.

RAG instructions (the system prompt the answering system was given):
<<<
{rag_instructions}
>>>

Criteria, each a true/false flag:
1. factuality: true only if every claim in the answer is supported by the retrieved chunks. False if any claim is missing from the chunks, contradicts them, or comes from outside knowledge. The fixed reply the instructions prescribe for unanswerable questions makes no claim, so it is true, unless the retrieved chunks clearly do contain the answer.
2. citation_accuracy: true only if the answer ends with the "Sources:" line the instructions require, every listed ID is among the retrieved chunks, and each listed chunk really contains the information the answer uses. False if a source is missing, invented, or does not support the answer. A prescribed refusal reply has no claims to cite, so it is true.
3. instruction_following: true only if the answer obeys ALL of these rules, which are the rules from the RAG instructions that matter:
   R1. For a protection-related question (online investment protection, banking account privacy, or fraud), the reply must be exactly: "Please refer to the protection section of your online investment for guidance."
   R2. An answer built from the context must end with a line of the form "Sources: <chunk ids you used>". The two exact replies in R1 and R3 are exempt from this rule.
   R3. If the answer is not in the retrieved chunks, the reply must be exactly: "Not specified in the agreement. Please contact your card issuer for more information at 123@gmail.com." It is a violation to use this reply when the chunks do contain the answer, and a violation to answer from outside the chunks when they do not.
   R4. The answer may only give guidance on credit card agreements.
   R5. The answer must never say "the document says", "the passage suggests" or similar wording that refers to the document or passage.
   R6. The tone must be polite and professional.
   The prescribed texts in R1 and R3 are required by the instructions, so they are never irrelevant content. Do not mark other things (format, length, completeness) as violations, because only R1-R6 count here.

For each criterion write the reasoning first, in one to three sentences that point to the evidence. For instruction_following the reasoning must say which rules were violated, if any. Then give the flag. Also list the violated rule numbers.

Return JSON with exactly these keys:
{"factuality_reasoning": "...", "factuality": true, "citation_accuracy_reasoning": "...", "citation_accuracy": true, "instruction_following_reasoning": "...", "instruction_following": true, "violated_rules": []}
violated_rules is a list of strings such as ["R2", "R5"], empty when instruction_following is true.

Question:
{question}

Retrieved chunks:
{context}

Answer:
{answer}'''

CRITERIA = (
    "factuality",
    "citation_accuracy",
    "instruction_following",
)

# Map paragraph IDs to the original indexed chunks.
chunk_by_id = {chunk.para_id: chunk for chunk in indexed_chunks}


def build_judge_prompt(row: dict) -> str:
    """Build the judging prompt without accidentally replacing text inside inputs."""
    retrieved_ids = row.get("retrieved_ids") or []

    retrieved_chunks = [
        chunk_by_id[chunk_id]
        for chunk_id in retrieved_ids
        if chunk_id in chunk_by_id
    ]

    context = format_context(retrieved_chunks)

    # Replace only the four original template placeholders.
    # Any placeholder-like text in inserted values remains untouched.
    replacements = {
        "{rag_instructions}": ANSWER_SYSTEM_PROMPT.strip(),
        "{question}": str(row.get("question", "")),
        "{context}": context,
        "{answer}": str(row.get("answer", "")),
    }

    # Use a single-pass substitution to avoid replacing placeholders
    # that happen to occur inside inserted question/context/answer text.
    import re

    pattern = re.compile(
        r"\{rag_instructions\}|\{question\}|\{context\}|\{answer\}"
    )

    return pattern.sub(
        lambda match: replacements[match.group(0)],
        JUDGE_PROMPT,
    )


def parse_verdict(payload: dict) -> dict:
    """Validate the judge response and normalize its output."""
    verdict = {}

    for name in CRITERIA:
        if name not in payload:
            raise ValueError(f"Missing judge field: {name}")

        if not isinstance(payload[name], bool):
            raise ValueError(
                f"{name} is not a boolean: {payload[name]!r}"
            )

        reasoning_key = f"{name}_reasoning"
        if reasoning_key not in payload:
            raise ValueError(f"Missing judge field: {reasoning_key}")

        verdict[name] = payload[name]
        verdict[reasoning_key] = str(payload[reasoning_key]).strip()

    violated = payload.get("violated_rules", [])

    if not isinstance(violated, list):
        raise ValueError(
            f"violated_rules is not a list: {violated!r}"
        )

    valid_rules = {"R1", "R2", "R3", "R4", "R5", "R6"}
    normalized_rules = [
        str(rule).strip().upper()
        for rule in violated
    ]

    unknown_rules = set(normalized_rules) - valid_rules
    if unknown_rules:
        raise ValueError(
            f"Unknown violated rules: {sorted(unknown_rules)}"
        )

    # A consistent list should not contain duplicates.
    verdict["violated_rules"] = sorted(
        set(normalized_rules),
        key=lambda rule: int(rule[1:]),
    )

    return verdict


def safe_judge(row: dict) -> tuple[dict | None, str | None]:
    """Judge one answer, retrying once on errors or malformed JSON."""
    error = None

    for _ in range(2):
        try:
            payload = judge_client.complete_json(
                build_judge_prompt(row),
                system=JUDGE_SYSTEM,
                temperature=0.0,
                max_tokens=1024,
            )

            return parse_verdict(payload), None

        except Exception as exc:  # noqa: BLE001
            error = f"{type(exc).__name__}: {exc}"[:250]

    return None, error


# Judge every evaluated answer, including in-scope and out-of-scope questions.
with ThreadPoolExecutor(max_workers=8) as pool:
    judged = list(pool.map(safe_judge, rows))


judge_failures = []

for row, (verdict, error) in zip(rows, judged, strict=True):
    row["judge"] = verdict

    if verdict is None:
        judge_failures.append((row["id"], error))
        console.print(f"  JUDGE FAILED {row['id']}: {error}")

console.print(
    f"Judged {sum(row['judge'] is not None for row in rows)} "
    f"of {len(rows)} answers; "
    f"{len(judge_failures)} judge failure(s)"
)

Judged 246 of 246 answers; 0 judge failure(s)

In [357]:

import json
import re
from collections import Counter
from statistics import mean

from rich.markup import escape
from rich.table import Table
from rouge_score import rouge_scorer


rouge = rouge_scorer.RougeScorer(["rougeL"])


def strip_sources(answer: str) -> str:
    """Remove a trailing Sources line and anything following it."""
    return re.sub(
        r"\s*Sources:.*$",
        "",
        answer,
        flags=re.IGNORECASE | re.DOTALL,
    ).strip()


def safe_mean(values: list[float]) -> float:
    """Return zero for empty collections."""
    return mean(values) if values else 0.0


def rate(selected: list[dict], name: str) -> str:
    """Share of successfully judged answers passing a criterion."""
    valid = [
        row for row in selected
        if row.get("judge") is not None
        and isinstance(row["judge"].get(name), bool)
    ]

    score = safe_mean([
        float(row["judge"][name])
        for row in valid
    ])

    return f"{score:.3f} ({len(valid)})"


# 1. Calculate ROUGE-L for in-scope answers with non-empty references.
in_scope_rows = [
    row for row in rows
    if row.get("in_scope", False)
]

for row in in_scope_rows:
    gold = str(row.get("gold_answer") or "").strip()
    answer = strip_sources(str(row.get("answer") or ""))

    row["rouge_l"] = (
        rouge.score(gold, answer)["rougeL"].fmeasure
        if gold and answer
        else 0.0
    )


# 2. Split by judgment status and scope.
judged_rows = [
    row for row in rows
    if row.get("judge") is not None
]

judged_in_scope = [
    row for row in judged_rows
    if row.get("in_scope", False)
]

judged_out = [
    row for row in judged_rows
    if not row.get("in_scope", False)
]

out_of_scope_rows = [
    row for row in rows
    if not row.get("in_scope", False)
]


# 3. Quality judge metrics.
table = Table(show_header=True, header_style="bold magenta")
table.add_column("Quality judge (share true, with n)")
table.add_column("In scope")
table.add_column("Out of scope")

table.add_row(
    "Factuality",
    rate(judged_in_scope, "factuality"),
    rate(judged_out, "factuality"),
)

table.add_row(
    "Citation accuracy",
    rate(judged_in_scope, "citation_accuracy"),
    rate(judged_out, "citation_accuracy"),
)

table.add_row(
    "Instruction following",
    rate(judged_in_scope, "instruction_following"),
    rate(judged_out, "instruction_following"),
)

console.print(table)


# 4. Rule violation counts.
violations = Counter(
    rule
    for row in judged_rows
    for rule in row["judge"].get("violated_rules", [])
)

console.print(
    "Instruction rule violations "
    "(R1 protection reply, R2 Sources line, "
    "R3 not-specified reply, R4 scope, R5 wording, R6 tone): "
    f"{dict(sorted(violations.items())) or 'none'}"
)


# 5. Other metrics.
table = Table(show_header=True, header_style="bold magenta")
table.add_column("Other metrics")
table.add_column("Value")

table.add_row(
    "ROUGE-L F (in-scope, mean)",
    f"{safe_mean([row['rouge_l'] for row in in_scope_rows]):.3f} "
    f"({len(in_scope_rows)})",
)

table.add_row(
    f"Refused, out of scope (target 1.0)",
    f"{safe_mean([float(bool(row.get('refused', False))) for row in out_of_scope_rows]):.3f} "
    f"({len(out_of_scope_rows)})",
)

table.add_row(
    "Refused, in scope (target 0.0)",
    f"{safe_mean([float(bool(row.get('refused', False))) for row in in_scope_rows]):.3f} "
    f"({len(in_scope_rows)})",
)

table.add_row(
    "Judge success rate",
    f"{len(judged_rows) / len(rows):.3f} ({len(judged_rows)}/{len(rows)})"
    if rows else "0.000 (0/0)",
)

console.print(table)


# 6. Display up to five answers with at least one failed criterion.
failed_rows = [
    row for row in judged_rows
    if not all(
        row["judge"].get(name) is True
        for name in CRITERIA
    )
]

for row in failed_rows[:5]:
    console.print(
        f"\n[bold]{escape(str(row.get('id', 'unknown')))}[/bold] "
        f"in_scope={row.get('in_scope')} "
        f"Q: {escape(str(row.get('question', '')))}"
    )

    console.print(f"A: {escape(str(row.get('answer', '')))}")

    for name in CRITERIA:
        verdict = row["judge"].get(name)
        reasoning = row["judge"].get(f"{name}_reasoning", "")
        console.print(
            f"  {name}={verdict}: {escape(str(reasoning))}"
        )

    console.print(
        f"  violated_rules={row['judge'].get('violated_rules', [])}"
    )


# 7. Save every row, including failed judgments.
with NAIVE_RESULTS_PATH.open("w", encoding="utf-8") as handle:
    for row in rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

console.print(
    f"\nSaved {len(rows)} results to {NAIVE_RESULTS_PATH}"
)


2026-10-29 22:30:48,674 INFO absl: Using default tokenizer.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Quality judge (share true, with n) ┃ In scope    ┃ Out of scope ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ Factuality                         │ 0.988 (246) │ 0.000 (0)    │
│ Citation accuracy                  │ 0.972 (246) │ 0.000 (0)    │
│ Instruction following              │ 0.967 (246) │ 0.000 (0)    │
└────────────────────────────────────┴─────────────┴──────────────┘

Instruction rule violations (R1 protection reply, R2 Sources line, R3 not-specified reply, R4 scope,
R5 wording, R6 tone): {'R1': 1, 'R2': 6, 'R3': 3}

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Other metrics                      ┃ Value           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ ROUGE-L F (in-scope, mean)         │ 0.633 (246)     │
│ Refused, out of scope (target 1.0) │ 0.000 (0)       │
│ Refused, in scope (target 0.0)     │ 0.061 (246)     │
│ Judge success rate                 │ 1.000 (246/246) │
└────────────────────────────────────┴─────────────────┘

naive-87f7e2d3-560d-405d-8df2-8c79591ffb3d-2 in_scope=True Q: What happens if you use a Mobile 
Payment App with your CIBC account?

...

factuality=True: The answer correctly states the conditions under which a statement might cover a 
three-month period, as specified in chunk [p0003]. The conditions listed in the answer match those 
found in the retrieved chunk.

citation_accuracy=False: The answer cites chunks p0003 and p0004, but only p0003 contains the 
relevant information about the conditions for a statement covering a three-month period. Therefore, 
the citation of p0004 is inaccurate.

instruction_following=False: The answer follows the instructions by providing information based on
the context and ending with a 'Sources:' line. However, it incorrectly cites chunk p0004, which does
not contain relevant information, violating R2.

violated_rules=['R2']

Saved 246 results to 
/home/coder/synthetic-data-bootcamp/implementations/RAG_QA_design/data/naive_results.jsonl

<pre> Save all the results in xlsx </pre>

In [358]:
import pandas as pd
from openpyxl.styles import Alignment

EXPORT_DIR = DATA_DIR / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def short(value):
    """Return the part after '::', or the original value if absent."""
    if not value:
        return ""

    parts = str(value).split("::", 1)
    return parts[1] if len(parts) > 1 else parts[0]


def export_id(row):
    """Shorten in-scope IDs while preserving out-of-scope IDs."""
    return short(row["id"]) if row["in_scope"] else row["id"]


# Build answer evaluation DataFrame
answers_df = pd.DataFrame([
    {
        "id": export_id(row),
        "in_scope": row["in_scope"],
        "question": row["question"],
        "rag_answer": row["answer"],
        "gold_answer": row["gold_answer"],
        "rouge_l": round(row["rouge_l"], 3)
        if row.get("rouge_l") is not None else None,
        "rag_words": len(strip_sources(row["answer"]).split()),
        "gold_words": len(row["gold_answer"].split()),
        "gold_chunk": short(row.get("gold_chunk")),
        "gold_rank": row.get("gold_rank"),
        "retrieved": ", ".join(
            short(chunk_id) for chunk_id in row.get("retrieved_ids", [])
        ),
        "refused": row["refused"],
    }
    for row in rows
])


# Build judge evaluation DataFrame
judge_df = pd.DataFrame([
    {
        "id": export_id(row),
        "in_scope": row["in_scope"],
        "question": row["question"],
        "rag_answer": row["answer"],
        "retrieved": ", ".join(
            short(chunk_id) for chunk_id in row.get("retrieved_ids", [])
        ),
        "factuality": row["judge"]["factuality"],
        "factuality_reasoning": row["judge"]["factuality_reasoning"],
        "citation_accuracy": row["judge"]["citation_accuracy"],
        "citation_accuracy_reasoning": row["judge"]["citation_accuracy_reasoning"],
        "instruction_following": row["judge"]["instruction_following"],
        "violated_rules": ", ".join(
            row["judge"].get("violated_rules", [])
        ),
        "instruction_following_reasoning": row["judge"]["instruction_following_reasoning"],
    }
    for row in judged_rows
])


def save_excel(frame: pd.DataFrame, path) -> None:
    """Save a DataFrame to a formatted Excel workbook."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        frame.to_excel(writer, index=False, sheet_name="Sheet1")
        sheet = writer.sheets["Sheet1"]

        # Keep the header visible while scrolling.
        sheet.freeze_panes = "A2"

        # Set column widths and wrap cell text.
        for column in sheet.columns:
            header = str(column[0].value or "")
            is_wide = (
                header in {"question", "rag_answer", "gold_answer"}
                or header.endswith("_reasoning")
            )

            sheet.column_dimensions[
                column[0].column_letter
            ].width = 60 if is_wide else 16

            for cell in column:
                cell.alignment = Alignment(
                    wrap_text=True,
                    vertical="top",
                )


# Export evaluation results
answers_path = EXPORT_DIR / "naive_answers.xlsx"
judge_path = EXPORT_DIR / "naive_judge.xlsx"

save_excel(answers_df, answers_path)
save_excel(judge_df, judge_path)

console.print(f"Saved {answers_path} ({len(answers_df)} rows)")
console.print(f"Saved {judge_path} ({len(judge_df)} rows)")


# Keep only rows with a ROUGE-L score
scored = answers_df[answers_df["rouge_l"].notna()].copy()

console.print(f"Rows with ROUGE-L scores: {len(scored)}")


Saved 
/home/coder/synthetic-data-bootcamp/implementations/RAG_QA_design/data/exports/naive_answers.xlsx 
(246 rows)

Saved 
/home/coder/synthetic-data-bootcamp/implementations/RAG_QA_design/data/exports/naive_judge.xlsx (246
rows)

Rows with ROUGE-L scores: 246